# 2-2. 벡터스토어 비교: Chroma / FAISS / Qdrant

⏱ **소요시간**: 30분

## 학습 목표
- 폐쇄망에서 쓸 수 있는 세 가지 벡터스토어의 특징과 제약사항을 구분할 수 있다.
- Chroma · FAISS · Qdrant의 영속화/인메모리 모드 사용법을 익힌다.
- HNSW 키 파라미터(M, efConstruction, efSearch)가 인덱싱/질의 속도와 정확도에 미치는 영향을 설명할 수 있다.
- 동일 코퍼스를 세 스토어에 적재해 **인덱싱 시간, 질의 시간, 결과 일치도**를 비교한다.

## 핵심 키워드
`Chroma` `FAISS` `Qdrant` `HNSW` `M` `efConstruction` `efSearch` `persist_directory` `save_local` `in-memory`

## 폐쇄망 적합성 배지
| 스토어 | 설치 | 폐쇄망 | 영속화 | 메타 필터 | 배지 |
|---|---|---|---|---|---|
| **Chroma** | `pip` 로컬 DB | 완전 OK | `persist_directory=...` | JSON 기반 $and/$or | 🔒 |
| **FAISS** | `faiss-cpu`, 메타 없음 | 완전 OK | `save_local / load_local` | 수동 필터링 | 🔒 |
| **Qdrant** | 인매모리 또는 도커 | 완전 OK | `:memory:` 또는 `path=...` | 풍부한 필터 DSL | 🔒 |
| ~~Pinecone / Weaviate Cloud~~ | 매니지드 SaaS | ❌ 불가 | — | — | ☁️ |

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

corpus_text = Path('../../data/corpus_ko.txt').read_text(encoding='utf-8')
# 실습에서 치수를 보려면 청크 개수가 충분해야 해 코퍼스를 인위적으로 복제
corpus_text = (corpus_text + '\n') * 5

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents([corpus_text], metadatas=[{'source': 'corpus_ko.txt'}])
for i, c in enumerate(chunks):
    c.metadata['chunk_id'] = i
    # 실습용 메타: 짝수 청크는 'finance', 홀수는 'compliance'
    c.metadata['topic'] = 'finance' if i % 2 == 0 else 'compliance'
print(f'인덱싱 대상 청크: {len(chunks)}개')
emb = get_embeddings()

## 1. Chroma 🔒

`persist_directory`를 지정하면 디스크에 SQLite + Parquet 형태로 영속화된다. 재시작 시도 재인덱싱 불필요.

In [ ]:
import time, shutil
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/chroma')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
chroma_vs = Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    persist_directory=str(CHROMA_DIR),
    collection_name='efinance_terms',
)
chroma_index_sec = time.time() - t0
print(f'[Chroma] index 시간: {chroma_index_sec:.3f}s, 문서 수: {chroma_vs._collection.count()}')

In [ ]:
q = '청약철회 기간은 며칠인가?'
t0 = time.time()
res = chroma_vs.similarity_search_with_score(q, k=3)
chroma_q_sec = time.time() - t0
print(f'[Chroma] query: {chroma_q_sec*1000:.1f}ms')
for doc, score in res:
    print(f'  score={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

## 2. FAISS 🔒

FAISS는 메타 저장소가 별도로 없고 동일 길이의 벡터 미진 중심이라 **인덱싱이 빠르고 메모리 효율적**이다. 필터링은 `filter=lambda m: ...`로 직접 처리해야 하는 불편이 있다.

In [ ]:
from langchain_community.vectorstores import FAISS

FAISS_DIR = Path('./_store/faiss')
if FAISS_DIR.exists():
    shutil.rmtree(FAISS_DIR)
FAISS_DIR.parent.mkdir(parents=True, exist_ok=True)

t0 = time.time()
faiss_vs = FAISS.from_documents(chunks, emb)
faiss_vs.save_local(str(FAISS_DIR))
faiss_index_sec = time.time() - t0
print(f'[FAISS] index 시간: {faiss_index_sec:.3f}s, 문서 수: {faiss_vs.index.ntotal}')

t0 = time.time()
res = faiss_vs.similarity_search_with_score(q, k=3)
faiss_q_sec = time.time() - t0
print(f'[FAISS] query: {faiss_q_sec*1000:.1f}ms')
for doc, score in res:
    print(f'  L2={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

In [ ]:
# FAISS 필터링 — 메타 topic=='finance' 인 청크만
res = faiss_vs.similarity_search(q, k=3, filter={'topic': 'finance'})
for d in res:
    print(f'  topic={d.metadata["topic"]} chunk_id={d.metadata["chunk_id"]}  {d.page_content[:60]}…')

## 3. Qdrant 🔒 (인메모리)

`QdrantClient(location=':memory:')`로 빠르게 프로토타이핑할 수 있고, 운영계에서는 `docker run qdrant/qdrant`로 겹섬례 배포한다. 풍부한 **payload filter DSL**이 장점.

In [ ]:
from langchain_community.vectorstores import Qdrant
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

# HNSW 파라미터를 명시적으로 지정
# - M: 각 노드가 가질 이웃 수 (높을수록 정확↑, 메모리↑)
# - ef_construct: 인덱싱 시 탐색 폭 (높을수록 인덱싱↑시간, 품질↑)
# - ef_search 는 질의 시에 hnsw_ef 로 넘긴다.
HNSW_M = 16
HNSW_EF_CONSTRUCT = 100
HNSW_EF_SEARCH = 64

client = QdrantClient(location=':memory:')
vec_dim = len(emb.embed_query('dim probe'))
client.recreate_collection(
    collection_name='efinance_terms',
    vectors_config=qmodels.VectorParams(size=vec_dim, distance=qmodels.Distance.COSINE),
    hnsw_config=qmodels.HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
)

t0 = time.time()
qdrant_vs = Qdrant(client=client, collection_name='efinance_terms', embeddings=emb)
qdrant_vs.add_documents(chunks)
qdrant_index_sec = time.time() - t0
print(f'[Qdrant] index 시간: {qdrant_index_sec:.3f}s (M={HNSW_M}, ef_construct={HNSW_EF_CONSTRUCT})')

In [ ]:
t0 = time.time()
res = qdrant_vs.similarity_search_with_score(q, k=3, search_params=qmodels.SearchParams(hnsw_ef=HNSW_EF_SEARCH))
qdrant_q_sec = time.time() - t0
print(f'[Qdrant] query: {qdrant_q_sec*1000:.1f}ms (ef_search={HNSW_EF_SEARCH})')
for doc, score in res:
    print(f'  cos={score:.3f} topic={doc.metadata["topic"]}  {doc.page_content[:60]}…')

In [ ]:
# Qdrant 필터 DSL — (topic=finance) AND (chunk_id ≥ 2)
flt = qmodels.Filter(must=[
    qmodels.FieldCondition(key='metadata.topic', match=qmodels.MatchValue(value='finance')),
    qmodels.FieldCondition(key='metadata.chunk_id', range=qmodels.Range(gte=2)),
])
res = qdrant_vs.similarity_search(q, k=3, filter=flt)
for d in res:
    print(f'  chunk_id={d.metadata["chunk_id"]} topic={d.metadata["topic"]}  {d.page_content[:60]}…')

## 4. HNSW 파라미터 감잡

HNSW(Hierarchical Navigable Small World)는 근사 최근접(Approximate Nearest Neighbor) 그래프를 계층으로 구성한다. 세 가지 파라미터만 잡으면 된다.

| 파라미터 | 시점 | 높이면 | 추천 범위 |
|---|---|---|---|
| **M** | 인덱싱 | recall↑, 메모리↑ | 8 ∼ 48 (기본 16) |
| **ef_construct** | 인덱싱 | recall↑, 인덱싱시간↑ | 100 ∼ 400 |
| **ef_search** (hnsw_ef) | 질의 | recall↑, 질의시간↑ | 32 ∼ 256 |

일반적으로 **M은 고정하고 ef_search만 튀닝**하면 충분하다. 아래에서 ef_search 변화에 따른 질의 시간을 측정해본다.

In [ ]:
import statistics

queries = [
    '청약철회 기간은?',
    '분실신고 절차는?',
    '망분리란 무엇인가?',
    'RAG의 정의는?',
    '지식그래프와 GraphRAG의 차이는?',
]

def bench_qdrant(ef):
    times = []
    for query in queries:
        t0 = time.time()
        qdrant_vs.similarity_search(query, k=3, search_params=qmodels.SearchParams(hnsw_ef=ef))
        times.append((time.time() - t0) * 1000)
    return statistics.mean(times), statistics.pstdev(times)

for ef in [16, 64, 128, 256]:
    mean_ms, std_ms = bench_qdrant(ef)
    print(f'  ef_search={ef:4d}  avg={mean_ms:5.1f}ms  ±{std_ms:.1f}ms')

## 5. 세 스토어 종합 비교

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'store': 'Chroma', 'index_sec': round(chroma_index_sec, 3), 'query_ms': round(chroma_q_sec*1000, 1), 'persist': 'persist_directory', 'filter': 'JSON where/and/or'},
    {'store': 'FAISS',  'index_sec': round(faiss_index_sec, 3),  'query_ms': round(faiss_q_sec*1000, 1),  'persist': 'save_local', 'filter': 'dict or lambda'},
    {'store': 'Qdrant', 'index_sec': round(qdrant_index_sec, 3), 'query_ms': round(qdrant_q_sec*1000, 1), 'persist': ':memory: or path=', 'filter': 'rich Filter DSL'},
])
summary

## 정리

- **빠른 프로토타입**: Chroma `persist_directory` 사용. 설치도 심플.
- **대규모 읽기 속도**: FAISS. 메모리 효율이 좋고 질의 속도가 안정적.
- **운영 환경**: Qdrant. Docker 배포, 부질의러운 필터, 월간 스냅샷 백업이 심플.
- HNSW는 `ef_search`에서 품질/속도 균형을 맞정지면 된다. M/ef_construct는 인덱싱 시 한 번만 결정.

## 과제
1. 코퍼스를 10배로 복제해 청크 개수를 늘린 후 세 스토어의 **인덱싱 시간**을 다시 재보세요.
2. Qdrant의 `M`을 8 / 32 / 48로 바꿔 **메모리 사용량 vs recall**을 확인해보세요.
3. FAISS에 IVF 인덱스(`IVFFlat`, `IVFPQ`)를 적용하면 메모리가 얼마나 줄어들지 계산해보세요.

## 더 읽어보기
- teddylee777/langchain-kr — [09-VectorStore](https://github.com/teddylee777/langchain-kr/tree/main/09-VectorStore)
- [HNSW 논문 (Malkov & Yashunin, 2018)](https://arxiv.org/abs/1603.09320)
- [Qdrant HNSW Tuning Docs](https://qdrant.tech/documentation/concepts/indexing/#vector-index)
- [FAISS Wiki — Index Types](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)